# Blatt 3 – LR-Zerlegung einer allgemeinen Matrix

**Referenznotebook für die Übungsaufgabe**  
Python 3 und NumPy

Wir berechnen die LR-Zerlegung $A=LR$ ohne Pivotisierung. Die untere Dreiecksmatrix $L$ hat Einsen auf der Diagonalen; die Matrix $R$ ist eine obere Dreiecksmatrix. Die Eliminationsfaktoren $l_{ik}$ werden in $L$ gespeichert.

Voraussetzung: Alle während der Elimination auftretenden Pivotelemente sind ungleich null. Für allgemeine Matrizen ist in der Praxis oft eine Pivotisierung erforderlich; sie ist **nicht** Bestandteil dieses Referenzbeispiels.

In [ ]:
import numpy as np


## 1. Allgemeine LR-Zerlegung

Im Eliminationsschritt $k$ gilt für $i>k$:

$$
l_{ik}=\frac{r^{(k)}_{ik}}{r^{(k)}_{kk}},
\qquad Z_i\leftarrow Z_i-l_{ik}Z_k.
$$

Die Einträge unterhalb der Diagonalen werden in $R$ zu null; die entsprechenden Multiplikatoren stehen in $L$.

In [ ]:
def factorize_lr(A):
    """LR-Zerlegung ohne Pivotisierung: A = L @ R.

    Parameter
    ---------
    A : quadratische Matrix

    Rueckgabe
    ---------
    L : untere Dreiecksmatrix mit Einsen auf der Diagonalen
    R : obere Dreiecksmatrix

    Hinweis: Fuer allgemeine Matrizen kann Pivotisierung notwendig sein.
    """
    R = np.array(A, dtype=float, copy=True)
    if R.ndim != 2 or R.shape[0] != R.shape[1]:
        raise ValueError("A muss eine quadratische Matrix sein.")

    n = R.shape[0]
    if n == 0:
        raise ValueError("A darf nicht leer sein.")

    L = np.eye(n, dtype=float)
    for k in range(n - 1):
        pivot = R[k, k]
        if pivot == 0.0:
            raise ZeroDivisionError(
                f"Nullpivot im Eliminationsschritt {k + 1}."
            )
        for i in range(k + 1, n):
            L[i, k] = R[i, k] / pivot
            R[i, k:] -= L[i, k] * R[k, k:]
            R[i, k] = 0.0

    if R[-1, -1] == 0.0:
        raise ZeroDivisionError("Nullpivot im letzten Diagonaleintrag.")
    return L, R


## 2. Vorwärts- und Rückwärtseinsetzen

Wir lösen $Lz=f$ und danach $Rx=z$.

In [ ]:
def forward_substitution(L, f):
    """Loest L @ z = f fuer eine untere Dreiecksmatrix L."""
    L = np.asarray(L, dtype=float)
    f = np.asarray(f, dtype=float)
    n = L.shape[0]
    if L.shape != (n, n) or f.shape != (n,):
        raise ValueError("Unpassende Dimensionen von L und f.")
    z = np.empty(n, dtype=float)
    for i in range(n):
        if L[i, i] == 0.0:
            raise ZeroDivisionError("Null auf der Diagonalen von L.")
        z[i] = (f[i] - np.dot(L[i, :i], z[:i])) / L[i, i]
    return z


def backward_substitution(R, z):
    """Loest R @ x = z fuer eine obere Dreiecksmatrix R."""
    R = np.asarray(R, dtype=float)
    z = np.asarray(z, dtype=float)
    n = R.shape[0]
    if R.shape != (n, n) or z.shape != (n,):
        raise ValueError("Unpassende Dimensionen von R und z.")
    x = np.empty(n, dtype=float)
    for i in range(n - 1, -1, -1):
        if R[i, i] == 0.0:
            raise ZeroDivisionError("Null auf der Diagonalen von R.")
        x[i] = (z[i] - np.dot(R[i, i + 1:], x[i + 1:])) / R[i, i]
    return x


## 3. Beispiel: 4 × 4

Das folgende Beispiel entspricht der LR-Aufgabe aus Blatt 3. Überprüfen Sie die Zerlegung $A=LR$ und die Lösung von $Ax=f$.

In [ ]:
A = np.array([
    [ 2.0, 1.0, 0.0,  1.0],
    [ 2.0, 3.0, 1.0,  0.0],
    [-2.0, 3.0, 4.0, -2.0],
    [ 4.0, 0.0, 1.0,  6.0]
])
f = np.array([5.0, 7.0, -2.0, 9.0])

L, R = factorize_lr(A)
z = forward_substitution(L, f)
x = backward_substitution(R, z)

print("L =\n", L)
print("R =\n", R)
print("z =", z)
print("x =", x)
print("||A - L@R||_inf =", np.linalg.norm(A - L @ R, ord=np.inf))
print("||Ax - f||_inf  =", np.linalg.norm(A @ x - f, ord=np.inf))

assert np.allclose(L @ R, A)
assert np.allclose(x, [1.0, 2.0, -1.0, 1.0])
assert np.allclose(A @ x, f)



## 4. Überleitung zur tridiagonalen LR-Zerlegung

Ist eine tridiagonale Matrix ohne Pivotisierung LR-zerlegbar, so ist $L$ eine untere Bidiagonalmatrix mit Einsen auf der Hauptdiagonalen und $R$ eine obere Bidiagonalmatrix.

Daher genügen drei Vektoren zur Speicherung der LR-Zerlegung: die Unterdiagonale von $L$ sowie die Haupt- und Oberdiagonale von $R$.